# QuantumXCT — object-oriented pipeline

A minimal, self-contained showcase of the `qxct` object-oriented API.

It runs on synthetic 2-genes-per-cell histograms, so it needs **no dataset and no real hardware**. Replace `synthetic_data()` with your own joint-histogram count dictionaries (and pass a real IBM backend to `assert_on_hardware`) to use it for real.

The whole run is configured once with a handful of dataclasses and driven through a single `QuantumXct` object.

In [ ]:
import numpy as np

from qxct import (
    QuantumXct, RunConfig,
    EncodingConfig, BackendConfig, SearchConfig, OptimizerConfig, MitigationConfig,
)

## 1. Data

Raw joint-histogram counts for two 2-gene cell types, basal vs. interacting. Qubit layout: cell 1 → q0, q1 ; cell 2 → q2, q3.

In [ ]:
ct1_basal = {"00": 80, "01": 10, "10": 10, "11": 0}
ct2_basal = {"00": 70, "01": 5,  "10": 5,  "11": 20}
ct1_inter = {"00": 20, "01": 40, "10": 30, "11": 10}
ct2_inter = {"00": 10, "01": 30, "10": 20, "11": 40}
genes = ["L1", "R1", "L2", "R2"]

## 2. Configure the whole run in one place

Every field has a default, so `RunConfig()` alone is valid. `EncodingConfig.norm` is the single L1/L2 switch (`'l1'` is the default, frequency-preserving encoding).

In [ ]:
config = RunConfig(
    encoding=EncodingConfig(norm="l1"),                 # 'l1' (default) or 'l2'
    backend=BackendConfig(backend=None, shots=2000),    # None -> local AerSimulator
    search=SearchConfig(
        algorithm="multi_epoch",                        # or 'iterative_n_wise'
        candidate_scope="all",                          # 'all' (GRN+LR) | 'intra' | 'inter'
        threshold=0.02, search_mode="abs",
        n_epochs=None, kl_tol=0.02, max_greedy_depth=6,
    ),
    optimizer=OptimizerConfig(method="COBYLA", initial_angle=np.pi / 2),
    mitigation=MitigationConfig(                        # only used by assert_on_hardware()
        dynamical_decoupling=True,
        twirling_gates=True, twirling_measure=True,
        measurement_mitigation=True, resilience_level=1, shots=4000,
    ),
)
config

## 3. Run the pipeline

`qx.run()` does: `screen_candidates` → `search_topology` → `optimize_angles` → `analyze`.

In [ ]:
qx = QuantumXct(config, gene_list=genes)
qx.encode(ct1_basal, ct2_basal, ct1_inter, ct2_inter)

result = qx.run()
print("Best topology :", result.topology)
print("CRX angles    :", [round(a, 3) for a in result.angles])
print("Final KL sum  :", round(result.kl_sum, 4))

The ablation study produced by `analyze()` is available as a DataFrame:

In [ ]:
result.contribution_df

## 4. Validate on real hardware

The search/optimization runs on a simulator. Once you have the best topology and angles, run that single circuit **once** on a real device to confirm the result survives device noise.

```python
from qiskit_ibm_runtime import QiskitRuntimeService
backend = QiskitRuntimeService().least_busy(operational=True, simulator=False)
hw = qx.assert_on_hardware(backend=backend)   # uses config.mitigation
```

Here we use the local Aer simulator as a stand-in (the mitigation flags are harmless no-ops on a simulator).

In [ ]:
hw = qx.assert_on_hardware()
print("Hardware 'assert quality' KL total:", round(hw["kl_total"], 4), "on backend", hw["backend"])
print("mitigation used:", hw["mitigation"])

## 5. Save results

Writes a JSON bundle (topology, angles, gene list, metadata) and, if an ablation table exists, a CSV alongside it.

In [ ]:
qx.save("results/example_run")

## Compare all search algorithms

Runs each `SearchConfig.algorithm` on the *same* data and tabulates gates / final KL / reduction, so you can compare **multi_epoch**, **iterative_n_wise** and **vqe_hybrid** side by side — the OO equivalent of the original script's separate search blocks. For this quick showcase `vqe_niter` is capped so the cell finishes fast.

`vqe_hybrid` builds an EfficientSU2 ansatz with one qubit per candidate CNOT, so it is the heaviest variant. Each run is kept in `algo_runs[...]` for further inspection (topology, angles, `assert_on_hardware`, `contribution_df`).

In [ ]:
import dataclasses
import pandas as pd
from qxct import baseline_kl

ALGORITHMS = ["multi_epoch", "iterative_n_wise", "vqe_hybrid"]
algo_runs, rows = {}, []
for algo in ALGORITHMS:
    s = dataclasses.replace(config.search, algorithm=algo)
    if algo == 'vqe_hybrid':
        s = dataclasses.replace(s, vqe_niter=50)
    cfg_a = dataclasses.replace(config, search=s)
    qx_a = QuantumXct(cfg_a, gene_list=genes)
    qx_a.encode(ct1_basal, ct2_basal, ct1_inter, ct2_inter)
    qx_a.screen_candidates()
    qx_a.search_topology()
    qx_a.optimize_angles()
    algo_runs[algo] = qx_a
    base = baseline_kl(qx_a.circ1, qx_a.circ2, qx_a.target1, qx_a.target2, cfg_a.backend.shots)
    rows.append({"algorithm": algo, "# gates": len(qx_a.result.topology),
                 "final KL": round(float(qx_a.result.kl_sum), 4),
                 "reduction": round(float(base - qx_a.result.kl_sum), 4),
                 "topology": qx_a.result.topology})

compare_df = pd.DataFrame(rows)
compare_df